In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Data extracted from your LaTeX tables
# -----------------------------
experiments = ["Binary", "Multiclass", "TGA (naive)", "TGA (RAG)", "Few-shot + Aug"]

# Extraction Logic:
# 1. Linear Model: "Linear SVM" was selected as the best consistent linear model across tables 1-4.
# 2. Transformer: Best between DistilBERT and XLM-RoBERTa per experiment from tables 5-11.
#    - Binary: Data not in new tables -> kept original RoBERTa values.
#    - Multiclass: DistilBERT (Acc 0.910 > XLM-R 0.893)
#    - TGA (naive): DistilBERT (Acc 0.889 > XLM-R 0.887)
#    - TGA (RAG): XLM-RoBERTa (F1 0.844 > DistilBERT 0.831; Acc is tied 0.896)
#    - Few-shot + Aug: DistilBERT (Acc 0.872 > XLM-R 0.829) - Source: Low Regime + TGA

data = {
    "experiment": experiments,
    # Linear SVM (Best Linear Model)
    "linear_acc":   [0.897, 0.862, 0.871, 0.871, 0.808],
    "linear_f1":    [0.897, 0.783, 0.789, 0.791, 0.731],
    
    # Best Transformer (DistilBERT or XLM-R depending on the experiment)
    "trf_acc":      [0.911, 0.910, 0.889, 0.896, 0.872],
    "trf_f1":       [0.911, 0.749, 0.824, 0.844, 0.803],
}
df = pd.DataFrame(data)

# -----------------------------
# Plot helpers
# -----------------------------
def plot_comparison(df: pd.DataFrame,
                    metric_left: str,
                    metric_right: str,
                    title: str,
                    outfile_prefix: str,
                    palette: str = "green"):
    """
    Grouped bars per experiment comparing Linear SVM vs Best Transformer.
    palette: "green" for Accuracy, "blue" for F1.
    """
    if palette == "green":
        # Colors for Accuracy
        lin_color = "#66bb6a"   # Light Green for Linear
        trf_color = "#c8e6c9"   # Paler Green for Transformer
        edge = "#1b5e20"
    else: 
        # Colors for F1 (Blue palette)
        lin_color = "#42a5f5"   # Blue for Linear
        trf_color = "#bbdefb"   # Paler Blue for Transformer
        edge = "#1565c0"

    x = np.arange(len(df["experiment"]))
    width = 0.36

    fig = plt.figure(figsize=(10, 5.5), dpi=140)
    ax = plt.gca()

    # Left bars: Linear SVM
    y_left = df[metric_left].values
    bars_left = ax.bar(
        x - width/2, y_left, width,
        color=lin_color, edgecolor="black", linewidth=0.7,
        label="Linear SVM"
    )

    # Right bars: Best Transformer
    y_right = df[metric_right].values
    bars_right = ax.bar(
        x + width/2, y_right, width,
        hatch="//", edgecolor=edge, facecolor=trf_color, linewidth=1.0,
        label="DistilBERT"
    )

    # Axes and labels
    ax.set_title(title, pad=12)
    ax.set_xticks(x, df["experiment"], rotation=15, ha="right")
    ax.set_ylabel("Score")

    # Dynamic Y range to highlight differences
    all_vals = np.concatenate([y_left[~np.isnan(y_left)], y_right[~np.isnan(y_right)]])
    vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
    span = max(1e-6, vmax - vmin)
    margin = max(0.02, span * 0.15) # Slightly more margin for labels
    ax.set_ylim(max(0.0, vmin - margin), min(1.0, vmax + margin))

    # Grid & spines
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    # Legend (upper right)
    ax.legend(frameon=False, loc="upper right")

    # Annotate bars
    def annotate(bars):
        for b in bars:
            h = b.get_height()
            if np.isnan(h):
                continue
            ax.annotate(f"{h:.3f}",
                        xy=(b.get_x() + b.get_width()/2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=8)
    annotate(bars_left)
    annotate(bars_right)

    # Save
    Path("figs").mkdir(parents=True, exist_ok=True)
    png = f"figs/{outfile_prefix}.png"
    pdf = f"figs/{outfile_prefix}.pdf"
    fig.tight_layout()
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    return png, pdf


# -----------------------------
# Build the two charts
# -----------------------------

# Plot Accuracy
acc_png, acc_pdf = plot_comparison(
    df, "linear_acc", "trf_acc",
    title="Linear SVM vs DistilBERT • Accuracy (TEST, masked)",
    outfile_prefix="svm_vs_transformer_accuracy_masked",
    palette="green"
)

# Plot F1
f1_png, f1_pdf = plot_comparison(
    df, "linear_f1", "trf_f1",
    title="Linear SVM vs DistilBERT • Macro-F1 (TEST, masked)",
    outfile_prefix="svm_vs_transformer_macroF1_masked",
    palette="blue"
)

print(f"Plots created successfully:\nAccuracy: {acc_png}\nMacro-F1: {f1_png}")

Plots created successfully:
Accuracy: figs/svm_vs_transformer_accuracy_masked.png
Macro-F1: figs/svm_vs_transformer_macroF1_masked.png
